# D401 — Users, Roles, Permissions, and Governance in Snowflake


This lesson builds a small, disposable training environment. You will create roles, grant privileges, assign roles to users, switch active roles, test access, revoke access, disable a demonstration user, and clean up. An optional extension connects roles to masking and row access policies.

The examples use synthetic data. The core lab uses standard access-control capabilities; the optional masking and row-filtering extension requires Enterprise Edition or higher.

## 1. What governance means in Snowflake

**Data governance** defines who is responsible for data, why it may be used, how it is protected, and how its lifecycle is managed. **Authentication** establishes identity. **Authorization** determines permitted actions.

A **user** is a Snowflake identity for a person or service. A **role** groups privileges. A **privilege** permits a particular operation, such as reading a table. A **securable object** is something access can be controlled on: a warehouse, database, schema, table, or policy.

A useful design path is:

**Approved business purpose → custom role → minimum object privileges → approved user assignment → role activation → verification and review.**

For this lab, an analyst reads orders, while a writer can also change them. In a real team, a data owner approves the purpose, a security administrator manages assignments, and engineers test that the intended access works.

Roles are one governance layer. Classification and tags describe information; masking controls visible values; row policies control visible records; retention and audit processes complete the lifecycle. A role name or a descriptive comment does not enforce those other requirements automatically.

## 2. Access-control vocabulary

**Role-Based Access Control (RBAC)** assigns permissions through roles. **Discretionary Access Control (DAC)** uses object ownership to administer access. Snowflake also supports **User-Based Access Control (UBAC)**, including direct user privileges considered when all secondary roles are enabled. This lab uses RBAC for understandable group-based administration. [Access-control framework](https://docs.snowflake.com/en/user-guide/security-access-control-overview).

**Least privilege** means granting only necessary permissions. **Separation of duties** divides sensitive responsibilities, such as approving access and implementing grants. An **entitlement** is an approved access allowance.

**Personally Identifiable Information (PII)** identifies or can be linked to a person. **Multi-Factor Authentication (MFA)** uses more than one authentication factor. **Single Sign-On (SSO)** connects login to an identity provider. Authentication controls do not replace table permissions or privacy policies.

Keep these questions separate:

| Question | Mechanism |
|---|---|
| Who is connecting? | User and authentication |
| Which job permissions apply? | Assigned and active roles |
| Can the table be read? | Object privileges |
| Which records and values are returned? | Row and masking policies |
| Is the use justified and time-limited? | Business approval and lifecycle governance |

## 3. All system-defined administrative roles

The five ordinary-account roles are listed first. Organization roles have account-specific availability.

| Role | Expansion | Built-in authority |
|---|---|---|
| `ACCOUNTADMIN` | Account Administrator | Inherits `SYSADMIN` and `SECURITYADMIN`; top account administration |
| `SECURITYADMIN` | Security Administrator | Global `MANAGE GRANTS`; inherits `USERADMIN` |
| `USERADMIN` | User and Role Administrator | `CREATE USER`, `CREATE ROLE`; manages identities and roles it owns |
| `SYSADMIN` | System Administrator | Creates databases and warehouses; manages objects through ownership and inherited privileges |
| `PUBLIC` | Public pseudo-role | Automatically available to every user and role; its grants are broadly available |
| `ORGADMIN` | Organization Administrator | Organization operations from enabled regular accounts; being phased out |
| `GLOBALORGADMIN` | Global Organization Administrator | Organization administration from the organization account only |

System-defined roles cannot be dropped, and Snowflake-provided grants on them cannot be revoked. [System-defined roles](https://docs.snowflake.com/en/user-guide/security-access-control-overview#system-defined-roles).

`GLOBALORGADMIN` manages organization-wide account lifecycle and usage. It is not expected in every regular account. `ORGADMIN` is separate from the ordinary account-role hierarchy; Snowflake recommends migration to the organization-account model. [Organization administrators](https://docs.snowflake.com/en/user-guide/organization-administrators).

## 4. What those permissions do—and how to inspect them

`MANAGE GRANTS` permits grant administration; it does not itself supply table-reading or object-creation privileges. `CREATE USER` does not mean ownership of every existing user. `SYSADMIN` needs an appropriate ownership or inheritance path to manage custom-owned objects. Use custom roles under an intentional hierarchy and keep routine work out of `ACCOUNTADMIN`. [Privilege definitions](https://docs.snowflake.com/en/user-guide/security-access-control-privileges); [role design best practices](https://docs.snowflake.com/en/user-guide/security-access-control-considerations).

These read-only statements show the actual grants in your account. Run as an authorized administrator for broad visibility:

```sql
USE ROLE SECURITYADMIN;
SHOW ROLES;
SHOW GRANTS TO ROLE ACCOUNTADMIN;
SHOW GRANTS TO ROLE SECURITYADMIN;
SHOW GRANTS TO ROLE USERADMIN;
SHOW GRANTS TO ROLE SYSADMIN;
SHOW GRANTS TO ROLE PUBLIC;
```

`SHOW GRANTS TO ROLE` shows direct grants; inspect child roles too when reasoning about inherited privileges. Actual grants can include account-specific additions. [SHOW GRANTS](https://docs.snowflake.com/en/sql-reference/sql/show-grants).

Snowflake also provides feature-specific database and application roles. They are not additional universal account administrator roles. For example, the `SNOWFLAKE` database contains database roles whose availability follows the product and account:

```sql
-- Read-only inventory; requires appropriate visibility of SNOWFLAKE.
SHOW DATABASE ROLES IN DATABASE SNOWFLAKE;
```

The listing is more reliable than treating a fixed feature-role list as universal. [SHOW DATABASE ROLES](https://docs.snowflake.com/en/sql-reference/sql/show-database-roles).

## 5. Role hierarchy: which direction do permissions flow?

**Role inheritance** lets a parent role receive its child role's permissions. We will build:

```text
ACCOUNTADMIN
├── SECURITYADMIN
│   └── USERADMIN
└── SYSADMIN
    └── D401_WRITER
        └── D401_READER

PUBLIC is implicitly available to every user and role.
```

The statement `GRANT ROLE D401_READER TO ROLE D401_WRITER` means the writer inherits the reader. It does not give readers writer access. A role can also have multiple parents; inspect all paths when removing access. [GRANT ROLE](https://docs.snowflake.com/en/sql-reference/sql/grant-role).

Role ownership and role inheritance differ. Creating or owning a role does not automatically make its data privileges usable. The lab explicitly grants its roles into the hierarchy and to the instructor's user.

## 6. Lab prerequisites and execution plan

Use a regular training account and an instructor login authorized to activate `USERADMIN`, `SECURITYADMIN`, and `SYSADMIN`. If an intern lacks these roles, the instructor runs administrative sections; do not give interns administrator roles merely to complete this exercise.

Run the core sections in order in the **same SQL worksheet session**. A session is the connection context retaining variables and active roles. If it resets, rerun the current-user variable assignment before commands using it.

The lab creates:

| Object | Name | Purpose |
|---|---|---|
| Database | `D401_LAB_DB` | Isolated training data |
| Schema | `ACCESS_LAB` | Managed grant administration |
| Warehouse | `D401_LAB_WH` | Small compute resource |
| Roles | `D401_READER`, `D401_WRITER` | Read and write access |
| User | `D401_DEMO_USER` | Disabled identity for assignment practice |

A **virtual warehouse** supplies compute. Its queries consume credits. This one starts suspended, automatically resumes for work, and automatically suspends after 60 seconds of inactivity. Cleanup removes it.

Check for naming collisions first. `SHOW` patterns use wildcard matching, so examine returned names rather than assume every match is exact:

```sql
USE ROLE SECURITYADMIN;
SHOW ROLES LIKE 'D401%';
SHOW USERS LIKE 'D401%';
USE ROLE SYSADMIN;
SHOW DATABASES LIKE 'D401%';
SHOW WAREHOUSES LIKE 'D401%';
```

If the named objects already belong to another exercise, choose a different prefix consistently. Setup intentionally uses `CREATE`, not replacement commands, so it stops on existing names.

## 7. Create custom roles

The active role needs `CREATE ROLE` on the account. `USERADMIN` supplies it. A comment documents intent but grants no access. [CREATE ROLE](https://docs.snowflake.com/en/sql-reference/sql/create-role).

```sql
USE ROLE USERADMIN;

CREATE ROLE D401_READER
  COMMENT = 'D401 lab: read synthetic orders';

CREATE ROLE D401_WRITER
  COMMENT = 'D401 lab: read and maintain synthetic orders';

SHOW ROLES LIKE 'D401%';
```

Expected: two new roles. They do not yet have permission to read the lab table, and no one has been assigned them through this setup yet.

Use meaningful names in real projects, such as a business function and environment. Review the actual grants rather than trusting a name such as `READ_ONLY`.

## 8. Create a demonstration user

`TYPE = PERSON` represents a human identity. This example creates a disabled user without a password, so no credential is embedded in the notebook. It is for administration practice, not a second login. The instructor tests data access using their existing authenticated session. [CREATE USER](https://docs.snowflake.com/en/sql-reference/sql/create-user).

```sql
USE ROLE USERADMIN;

CREATE USER D401_DEMO_USER
  TYPE = PERSON
  DISABLED = TRUE
  DEFAULT_ROLE = D401_READER
  COMMENT = 'D401 disabled demonstration identity; no login configured';

DESCRIBE USER D401_DEMO_USER;
```

`DEFAULT_ROLE` is a session preference, not a grant. Setting it does not assign the role. A real onboarding workflow also configures the organization's approved authentication method and any required MFA. Do not assume password-only login will satisfy the account's authentication policies.

## 9. Assign roles to users and to other roles

Snowflake expresses membership as **grant the role to the user**. “Add a user to a role” describes the same relationship in ordinary language; there is no separate reverse membership command.

`IDENTIFIER()` interprets a string or variable as an object name. Here it uses the current login's exact user name, including names that need quoting, without a hard-coded username. [Identifier variables](https://docs.snowflake.com/en/sql-reference/identifier-literal).

```sql
USE ROLE SECURITYADMIN;
SET D401_ACTOR = CURRENT_USER();

GRANT ROLE D401_READER TO ROLE D401_WRITER;
GRANT ROLE D401_WRITER TO ROLE SYSADMIN;

GRANT ROLE D401_READER TO USER D401_DEMO_USER;

-- Assign both so the instructor can explicitly activate either one.
GRANT ROLE D401_READER TO USER IDENTIFIER($D401_ACTOR);
GRANT ROLE D401_WRITER TO USER IDENTIFIER($D401_ACTOR);

SHOW GRANTS TO USER D401_DEMO_USER;
SHOW GRANTS OF ROLE D401_READER;
```

Expected: the reader is assigned to the demonstration user, the instructor, and the writer role. The writer is assigned to `SYSADMIN` and the instructor. `OF ROLE` identifies recipients; `TO ROLE` identifies privileges and child-role grants received.

## 10. Create the training objects

A **database** contains schemas. A **schema** groups tables and other objects. A **managed access schema** centralizes object-grant decisions in the schema owner or a role with `MANAGE GRANTS`; individual object owners cannot independently distribute access there. [CREATE SCHEMA](https://docs.snowflake.com/en/sql-reference/sql/create-schema).

```sql
USE ROLE SYSADMIN;
USE SECONDARY ROLES NONE;

CREATE WAREHOUSE D401_LAB_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

CREATE DATABASE D401_LAB_DB;
CREATE SCHEMA D401_LAB_DB.ACCESS_LAB WITH MANAGED ACCESS;

USE WAREHOUSE D401_LAB_WH;

CREATE TABLE D401_LAB_DB.ACCESS_LAB.ORDERS (
  ORDER_ID INTEGER,
  REGION VARCHAR,
  CUSTOMER_EMAIL VARCHAR,
  AMOUNT NUMBER(10, 2)
);

INSERT INTO D401_LAB_DB.ACCESS_LAB.ORDERS
  (ORDER_ID, REGION, CUSTOMER_EMAIL, AMOUNT)
VALUES
  (101, 'WEST', 'learner1@example.invalid', 120.00),
  (102, 'EAST', 'learner2@example.invalid', 250.00);
```

Expected: two synthetic records. Objects are owned by the creating primary role, `SYSADMIN`. Readers will receive use and read access, not ownership. Warehouse options are documented in [CREATE WAREHOUSE](https://docs.snowflake.com/en/sql-reference/sql/create-warehouse).

## 11. Grant privileges to roles

Reading this table requires a complete access path: warehouse `USAGE`, database `USAGE`, schema `USAGE`, and table `SELECT`. A permission on one layer does not replace the others.

```sql
USE ROLE SECURITYADMIN;

GRANT USAGE ON WAREHOUSE D401_LAB_WH TO ROLE D401_READER;
GRANT USAGE ON DATABASE D401_LAB_DB TO ROLE D401_READER;
GRANT USAGE ON SCHEMA D401_LAB_DB.ACCESS_LAB TO ROLE D401_READER;
GRANT SELECT ON TABLE D401_LAB_DB.ACCESS_LAB.ORDERS
  TO ROLE D401_READER;

GRANT INSERT, UPDATE, DELETE
  ON TABLE D401_LAB_DB.ACCESS_LAB.ORDERS
  TO ROLE D401_WRITER;

SHOW GRANTS TO ROLE D401_READER;
SHOW GRANTS TO ROLE D401_WRITER;
```

The writer inherits the reader's grants, so its read path need not be duplicated. `INSERT` adds records, `UPDATE` changes them, and `DELETE` removes them. Neither role receives table creation or ownership.

`ALL PRIVILEGES` means privileges relevant to the specified object, subject to the grantor's authority, and excludes ownership. It is not access to every object in the account. `WITH GRANT OPTION` permits onward delegation; this lab does not grant it. [GRANT privileges](https://docs.snowflake.com/en/sql-reference/sql/grant-privilege).

## 12. Assume a role: activate it with USE ROLE

“Assume a role” means activating its authority in your session. Snowflake uses `USE ROLE`; it does not log in as another user. The **primary role** is the currently selected role. `DEFAULT_ROLE` influences new sessions, while `USE ROLE` changes the current session. [USE ROLE](https://docs.snowflake.com/en/sql-reference/sql/use-role).

**Secondary roles** can contribute other granted permissions. Disable them for a focused test. This does not remove privileges inherited below the primary role or implicitly available through `PUBLIC`. Object creation is authorized through the primary role and its hierarchy. [USE SECONDARY ROLES](https://docs.snowflake.com/en/sql-reference/sql/use-secondary-roles).

```sql
USE ROLE D401_READER;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D401_LAB_WH;

SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_SECONDARY_ROLES();
SELECT * FROM D401_LAB_DB.ACCESS_LAB.ORDERS ORDER BY ORDER_ID;
```

Expected: the same instructor username, primary role `D401_READER`, and two records. At this point there is no masking policy: synthetic email values are visible.

Keeping the same user and changing the role makes it easier to understand the difference between identity and authority.

## 13. Negative test: readers cannot write

A **negative test** verifies that prohibited work fails. Run this block separately; its error is the expected outcome, so do not include it in an unattended “run all” batch.

```sql
USE ROLE D401_READER;
USE SECONDARY ROLES NONE;

-- EXPECTED ERROR: D401_READER has no INSERT privilege.
INSERT INTO D401_LAB_DB.ACCESS_LAB.ORDERS
VALUES (999, 'WEST', 'blocked@example.invalid', 1.00);
```

Expected: an insufficient-privileges error and no added row. If it succeeds, investigate extra grants or hierarchy paths before continuing. A role is not restricted merely because its name contains `READER`.

## 14. Positive test: writers can write and inherit reading

A **positive test** verifies that approved work succeeds.

```sql
USE ROLE D401_WRITER;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D401_LAB_WH;

INSERT INTO D401_LAB_DB.ACCESS_LAB.ORDERS
VALUES (103, 'WEST', 'learner3@example.invalid', 80.00);

UPDATE D401_LAB_DB.ACCESS_LAB.ORDERS
SET AMOUNT = 85.00 WHERE ORDER_ID = 103;

SELECT * FROM D401_LAB_DB.ACCESS_LAB.ORDERS ORDER BY ORDER_ID;

DELETE FROM D401_LAB_DB.ACCESS_LAB.ORDERS WHERE ORDER_ID = 103;
```

Expected: the temporary third record appears with amount 85.00, then is removed. The original two records remain for later exercises.

This permission design is for synthetic training data. A real “writer” job does not automatically justify full access to customer identities.

## 15. Existing-object grants versus future grants

An **existing-object grant** applies to objects already present. A **future grant** supplies permissions when new objects are created in a scope. One does not replace the other. Schema-level future grants for an object type take precedence over database-level future grants for that type. [Future grants](https://docs.snowflake.com/en/sql-reference/sql/grant-privilege#future-grants).

```sql
USE ROLE SECURITYADMIN;

GRANT SELECT ON ALL TABLES IN SCHEMA D401_LAB_DB.ACCESS_LAB
  TO ROLE D401_READER;

GRANT SELECT ON FUTURE TABLES IN SCHEMA D401_LAB_DB.ACCESS_LAB
  TO ROLE D401_READER;

SHOW FUTURE GRANTS IN SCHEMA D401_LAB_DB.ACCESS_LAB;

USE ROLE SYSADMIN;
CREATE TABLE D401_LAB_DB.ACCESS_LAB.NEXT_BATCH (BATCH_ID INTEGER);

USE ROLE D401_READER;
USE SECONDARY ROLES NONE;
SELECT * FROM D401_LAB_DB.ACCESS_LAB.NEXT_BATCH;
```

Expected: the new table is readable and empty. Its database and schema still require their own access grants.

Broad future grants also expose newly added tables. Use them within intentionally governed scopes; do not let unreviewed personal datasets arrive in a generally readable schema.

## 16. Revoke a permission and verify its effect

`REVOKE` removes a grant; it does not create an overriding denial. The same permission may remain through another role or grant path. [REVOKE privileges](https://docs.snowflake.com/en/sql-reference/sql/revoke-privilege).

```sql
USE ROLE SECURITYADMIN;
REVOKE INSERT ON TABLE D401_LAB_DB.ACCESS_LAB.ORDERS
  FROM ROLE D401_WRITER;
SHOW GRANTS TO ROLE D401_WRITER;
```

Run the following negative test separately:

```sql
USE ROLE D401_WRITER;
USE SECONDARY ROLES NONE;

-- EXPECTED ERROR: INSERT was revoked.
INSERT INTO D401_LAB_DB.ACCESS_LAB.ORDERS
VALUES (104, 'EAST', 'blocked2@example.invalid', 10.00);
```

Restore the lab's original writer permission:

```sql
USE ROLE SECURITYADMIN;
GRANT INSERT ON TABLE D401_LAB_DB.ACCESS_LAB.ORDERS
  TO ROLE D401_WRITER;
```

`UPDATE` and `DELETE` were not revoked. In production, consider dependent onward grants when choosing revocation behavior such as `RESTRICT` or `CASCADE`; this lab has no delegated grant chains.

## 17. Revoke future access and existing access separately

Removing a future-grant rule stops it applying to later objects. It does not erase permissions already materialized on existing tables. [Future-grant revocation](https://docs.snowflake.com/en/sql-reference/sql/revoke-privilege).

```sql
USE ROLE SECURITYADMIN;

REVOKE SELECT ON FUTURE TABLES IN SCHEMA D401_LAB_DB.ACCESS_LAB
  FROM ROLE D401_READER;

-- The existing table needs a separate revocation.
REVOKE SELECT ON TABLE D401_LAB_DB.ACCESS_LAB.NEXT_BATCH
  FROM ROLE D401_READER;

SHOW FUTURE GRANTS IN SCHEMA D401_LAB_DB.ACCESS_LAB;
SHOW GRANTS ON TABLE D401_LAB_DB.ACCESS_LAB.NEXT_BATCH;
```

Expected: no reader future-table rule and no reader `SELECT` on `NEXT_BATCH`. The `ORDERS` read grant remains. This illustrates why offboarding must inspect both current permissions and automation that creates future permissions.

## 18. Remove a user-role assignment and restore it

Removing membership keeps both the user and the role. It removes that specific grant edge. The person may still inherit the role through a different assigned parent. [REVOKE ROLE](https://docs.snowflake.com/en/sql-reference/sql/revoke-role).

```sql
USE ROLE SECURITYADMIN;

REVOKE ROLE D401_READER FROM USER D401_DEMO_USER;
SHOW GRANTS TO USER D401_DEMO_USER;

GRANT ROLE D401_READER TO USER D401_DEMO_USER;
SHOW GRANTS TO USER D401_DEMO_USER;
```

The demonstration user is disabled throughout; these commands practice membership administration without another login.

Changing a default role does not assign it, and removing an assignment does not mean a different inherited access path disappears. Review both preferences and grants during onboarding or offboarding.

## 19. Can a Snowflake role be suspended?

Snowflake has **no `SUSPEND ROLE` or role `DISABLED` property**. `ALTER ROLE` supports operations such as renaming and updating a comment; it is not a login-control command. [ALTER ROLE](https://docs.snowflake.com/en/sql-reference/sql/alter-role).

To temporarily remove a role's availability, preserve its grants and revoke its assignments from users and parent roles. In this isolated lab, the writer has exactly two recipient paths:

```sql
USE ROLE SECURITYADMIN;
SET D401_ACTOR = CURRENT_USER();

SHOW GRANTS OF ROLE D401_WRITER;
SHOW GRANTS TO ROLE D401_WRITER;

REVOKE ROLE D401_WRITER FROM USER IDENTIFIER($D401_ACTOR);
REVOKE ROLE D401_WRITER FROM ROLE SYSADMIN;

SHOW GRANTS OF ROLE D401_WRITER;
```

Expected: the writer still exists and retains its object privileges and child reader role, but has no recipients in this lab. We switched to `SECURITYADMIN` before changing the assignments.

Restore the exact lab paths:

```sql
USE ROLE SECURITYADMIN;
GRANT ROLE D401_WRITER TO ROLE SYSADMIN;
GRANT ROLE D401_WRITER TO USER IDENTIFIER($D401_ACTOR);
```

In a real account, enumerate every recipient and inherited path. This is access reassignment, not a native suspended state. For immediate user lockout and query interruption, use the user control in the next section.

## 20. Disable and re-enable a user

A **disabled user** cannot log in or initiate more queries; setting `DISABLED = TRUE` also aborts their running or scheduled statements. The identity and its grants remain. Re-enabling restores the possibility of login subject to configured authentication. [ALTER USER](https://docs.snowflake.com/en/sql-reference/sql/alter-user).

Only the disabled demonstration user is targeted here. Do not substitute your active instructor user.

```sql
USE ROLE USERADMIN;

ALTER USER D401_DEMO_USER SET DISABLED = FALSE;
DESCRIBE USER D401_DEMO_USER;

-- Return the credential-free demonstration identity to disabled state.
ALTER USER D401_DEMO_USER SET DISABLED = TRUE;
DESCRIBE USER D401_DEMO_USER;
```

No authentication method was configured for this demo identity; enabling it does not create a password or SSO mapping.

`ALTER USER ... ABORT ALL QUERIES` interrupts work without blocking future login, so it is a different operation. Suspending a warehouse stops compute availability; it is not a substitute for removing a user's access.

## 21. Modify and remove a custom role

Use a disposable role to practice renaming without disturbing the reader/writer lab:

```sql
USE ROLE USERADMIN;

CREATE ROLE D401_RETIRE_ME COMMENT = 'Disposable lifecycle example';
ALTER ROLE D401_RETIRE_ME SET COMMENT = 'Reviewed and ready for retirement';
ALTER ROLE D401_RETIRE_ME RENAME TO D401_RETIRED;

SHOW ROLES LIKE 'D401_RETIRED';
SHOW GRANTS TO ROLE D401_RETIRED;
SHOW GRANTS OF ROLE D401_RETIRED;

DROP ROLE D401_RETIRED;
```

Renaming does not revoke grants. If policy bodies or external configuration refer to role names as text, review those references separately.

`DROP ROLE` permanently removes the role. It revokes grants naming it as grantor or grantee; owned objects can transfer to the executing role rather than being deleted. Shared-database ownership has an additional transfer requirement. A role cannot be dropped while it is the current primary role. [DROP ROLE](https://docs.snowflake.com/en/sql-reference/sql/drop-role).

Before retiring a real role, inventory ownership, recipients, current and future grants, and policy references. Transfer ownership intentionally rather than rely on incidental transfer during removal. [GRANT OWNERSHIP](https://docs.snowflake.com/en/sql-reference/sql/grant-ownership).

## 22. Optional: database roles

An **account role** can receive privileges across databases. A **database role** packages privileges within one database and is granted to an account role for use. It cannot be activated directly with `USE ROLE`. This optional example gives the reader access to `NEXT_BATCH` through a database role. [CREATE DATABASE ROLE](https://docs.snowflake.com/en/sql-reference/sql/create-database-role); [GRANT DATABASE ROLE](https://docs.snowflake.com/en/sql-reference/sql/grant-database-role).

```sql
USE ROLE SYSADMIN;
CREATE DATABASE ROLE D401_LAB_DB.BATCH_READER;

GRANT USAGE ON DATABASE D401_LAB_DB
  TO DATABASE ROLE D401_LAB_DB.BATCH_READER;
GRANT USAGE ON SCHEMA D401_LAB_DB.ACCESS_LAB
  TO DATABASE ROLE D401_LAB_DB.BATCH_READER;
GRANT SELECT ON TABLE D401_LAB_DB.ACCESS_LAB.NEXT_BATCH
  TO DATABASE ROLE D401_LAB_DB.BATCH_READER;

GRANT DATABASE ROLE D401_LAB_DB.BATCH_READER TO ROLE D401_READER;

USE ROLE D401_READER;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D401_LAB_WH;
SELECT * FROM D401_LAB_DB.ACCESS_LAB.NEXT_BATCH;
```

Expected: the empty table is readable again. The warehouse privilege still comes from the account role. Dropping the lab database during cleanup removes its database role too.

## 23. Optional: enforce governance with masking and row policies

**Enterprise Edition or higher required.** Run after the core lab, with the two original order records present and writer assignments restored.

A **masking policy** controls a column's returned value. A **row access policy** controls record visibility. This example hides emails and shows only West records to the reader; the writer sees all synthetic data. The choice is a classroom rule, not a recommendation to expose real PII to every writer.

`IS_ROLE_IN_SESSION` checks role membership in the active role hierarchies, including secondary roles. Therefore `SYSADMIN` also qualifies for writer access in our hierarchy. [Role context function](https://docs.snowflake.com/en/sql-reference/functions/is_role_in_session).

```sql
USE ROLE SYSADMIN;
USE SECONDARY ROLES NONE;

CREATE MASKING POLICY D401_LAB_DB.ACCESS_LAB.EMAIL_MASK
  AS (V VARCHAR) RETURNS VARCHAR ->
    CASE WHEN IS_ROLE_IN_SESSION('D401_WRITER')
         THEN V ELSE '[hidden]' END;

ALTER TABLE D401_LAB_DB.ACCESS_LAB.ORDERS
  MODIFY COLUMN CUSTOMER_EMAIL
  SET MASKING POLICY D401_LAB_DB.ACCESS_LAB.EMAIL_MASK;

CREATE ROW ACCESS POLICY D401_LAB_DB.ACCESS_LAB.REGION_ACCESS
  AS (R VARCHAR) RETURNS BOOLEAN ->
    IS_ROLE_IN_SESSION('D401_WRITER')
    OR (IS_ROLE_IN_SESSION('D401_READER') AND R = 'WEST');

ALTER TABLE D401_LAB_DB.ACCESS_LAB.ORDERS
  ADD ROW ACCESS POLICY D401_LAB_DB.ACCESS_LAB.REGION_ACCESS ON (REGION);
```

`SYSADMIN` owns the schema, table, and policies in this lab, supplying the necessary creation and attachment authority. Real deployments can separate these responsibilities. [CREATE MASKING POLICY](https://docs.snowflake.com/en/sql-reference/sql/create-masking-policy); [CREATE ROW ACCESS POLICY](https://docs.snowflake.com/en/sql-reference/sql/create-row-access-policy).

## 24. Optional: compare policy results

Run only if the previous extension completed successfully:

```sql
USE ROLE D401_READER;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D401_LAB_WH;
SELECT * FROM D401_LAB_DB.ACCESS_LAB.ORDERS ORDER BY ORDER_ID;

USE ROLE D401_WRITER;
USE SECONDARY ROLES NONE;
SELECT * FROM D401_LAB_DB.ACCESS_LAB.ORDERS ORDER BY ORDER_ID;
```

| Active role | Visible order identifiers | Email result |
|---|---|---|
| Reader | 101 | `[hidden]` |
| Writer | 101 and 102 | Original synthetic values |

Base `SELECT` privileges still matter; policies do not grant table access. Masking leaves originals stored, and row filtering leaves hidden records stored. Keep secondary roles disabled so the instructor's broader roles do not contribute to this test.

Tags and classification can automate how policies are associated with sensitive columns, but this example attaches policies directly so their role-dependent behavior is easy to inspect.

## 25. Access reviews and troubleshooting

Use immediate grant inspection to investigate unexpected access:

```sql
USE ROLE SECURITYADMIN;
SHOW GRANTS TO ROLE D401_READER;
SHOW GRANTS TO ROLE D401_WRITER;
SHOW GRANTS OF ROLE D401_READER;
SHOW GRANTS OF ROLE D401_WRITER;
SHOW GRANTS TO USER D401_DEMO_USER;
SHOW GRANTS ON TABLE D401_LAB_DB.ACCESS_LAB.ORDERS;
SHOW FUTURE GRANTS IN SCHEMA D401_LAB_DB.ACCESS_LAB;
```

| Symptom | Check |
|---|---|
| `USE ROLE` fails | Assignment, role name, and session user |
| Reading fails despite `SELECT` | Database, schema, and warehouse use permissions |
| Reader can write | Secondary roles, inherited grants, and `PUBLIC` |
| Revocation seems ineffective | Alternate direct or inherited access paths |
| New tables are unexpectedly readable | Future grants and their scope |
| Policy test returns too much | Actual active role hierarchy and policy attachment |
| Variable is missing | Rerun `SET D401_ACTOR = CURRENT_USER()` in this session |
| Creation fails on an existing name | Inspect prior lab state; do not replace unrelated objects |

A real access review records the business purpose, approver, expiry, effective privileges, and evidence. Repeat it when people change jobs or datasets become more sensitive.

## 26. Cleanup: remove this lab only

Run after completing the desired exercises. These commands remove the training database and its contents, warehouse, demonstration user, and custom roles. They do not target system roles. Verify the names still refer to this lab before execution.

Dropping the database before the account roles also removes the optional policies and database role. The database is self-contained, so there are no external policy attachments to preserve in this design.

```sql
USE ROLE SYSADMIN;
USE SECONDARY ROLES NONE;
DROP DATABASE IF EXISTS D401_LAB_DB;
DROP WAREHOUSE IF EXISTS D401_LAB_WH;

USE ROLE USERADMIN;
DROP USER IF EXISTS D401_DEMO_USER;
DROP ROLE IF EXISTS D401_WRITER;
DROP ROLE IF EXISTS D401_READER;
DROP ROLE IF EXISTS D401_RETIRE_ME;
DROP ROLE IF EXISTS D401_RETIRED;

USE ROLE SECURITYADMIN;
SHOW ROLES LIKE 'D401%';
SHOW USERS LIKE 'D401%';
```

Expected: no objects from this lab remain in those role/user listings. Dropping these custom roles removes their user and hierarchy assignments. The worksheet remains on `SECURITYADMIN`; return to your normal approved working role when finished.

Database removal follows Snowflake recovery-retention behavior; it is not a promise of immediate physical erasure of every historical version.

## 27. Review questions and answers

1. **Does creating a role assign it to the creator?** No; explicitly grant it to a user or parent role.
2. **Does setting a default role grant it?** No; it is a preference.
3. **Does granting reader to writer make the reader a writer?** No; privileges flow upward to the recipient writer role.
4. **Why disable secondary roles while testing?** To avoid unrelated assigned roles authorizing the operation.
5. **Does revoking one privilege guarantee denial?** No; other grant paths may remain.
6. **How do you suspend a role?** There is no native role suspension. Remove its recipient paths or relevant privileges; disable a user for user lockout.
7. **Does disabling a user remove their grants?** No; it blocks access while retaining the identity and grants.
8. **Does removing a future grant remove earlier table grants?** No; inspect and revoke existing permissions separately.
9. **Can `MANAGE GRANTS` alone read tables?** No; grant administration and data access differ.
10. **Why do masking and row policies complement roles?** Roles provide authority; policies refine permitted values and records.

## 28. Abbreviation glossary and references

| Abbreviation | Expansion |
|---|---|
| SQL | Structured Query Language |
| RBAC | Role-Based Access Control |
| DAC | Discretionary Access Control |
| UBAC | User-Based Access Control |
| PII | Personally Identifiable Information |
| MFA | Multi-Factor Authentication |
| SSO | Single Sign-On |

System role-name expansions appear in section 3. `DB` and `WH` in our object names abbreviate database and warehouse. SQL keywords are executable language tokens: their purpose is explained beside each exercise.

Official command references are linked throughout the notebook. The primary starting points are [Snowflake access control](https://docs.snowflake.com/en/user-guide/security-access-control-overview), [privilege definitions](https://docs.snowflake.com/en/user-guide/security-access-control-privileges), and [access-control best practices](https://docs.snowflake.com/en/user-guide/security-access-control-considerations).